# ECB Communications → Supervised Learning (TF‑IDF)

This notebook loads ECB statements and MRO policy rates, detects effective rate-change events, maps them to statement dates within a business-day window, and trains simple text models:
- TF‑IDF + Logistic Regression (elastic‑net) for **classification**
- TF‑IDF + ElasticNet for **regression** (optional)

The notebook includes robust CSV loading and date normalization. Adjust the file paths at the top as needed.

In [ ]:
# --- Setup
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import BDay

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.metrics import classification_report, confusion_matrix

# Optional helpers; minimal fallbacks if not available
try:
    from skfin.text import coefs_plot, show_text
    from skfin.plot import line
    _HAS_SKFIN = True
except Exception:
    _HAS_SKFIN = False
    def coefs_plot(df_coef, title="Coefficients"):
        s = df_coef.squeeze()
        top = s.nlargest(10)
        bot = s.nsmallest(10)
        fig, ax = plt.subplots(figsize=(8, 6))
        both = pd.concat([bot, top])
        both.sort_values().plot(kind="barh", ax=ax)
        ax.set_title(title)
        plt.tight_layout()

    def show_text(df_text, lexica=None, n=None):
        for idx, row in df_text.iterrows():
            print(f"=== {idx} ===")
            txt = str(row['text'])
            print(txt[:800] + ("..." if len(txt) > 800 else ""))
            if lexica:
                print("\nTop positive:\n", list(lexica.get("positive", []).index))
                print("Top negative:\n", list(lexica.get("negative", []).index))
            print()

    def line(df, sort=False, ax=None, title=None):
        ax = ax or plt.gca()
        df.plot(ax=ax)
        if title:
            ax.set_title(title)

# Paths: try absolute then fallback to ./data
ABS_DATA = Path(r"C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset")
REL_DATA = Path("./data")
DATA_DIR = ABS_DATA if ABS_DATA.exists() else REL_DATA

ECB_STATEMENTS_CSV = DATA_DIR / "ecb_speeches_clean_minimal.csv"
ECB_POLICY_RATES_CSV = DATA_DIR / "ecb_policy_rates_daily_fake.csv"   # replace with your real file if needed
EUROSTOXX_CSV       = DATA_DIR / "eurostoxx_daily_fake.csv"           # optional

print('[PATHS]')
print('DATA_DIR         :', DATA_DIR)
print('STATEMENTS exists:', ECB_STATEMENTS_CSV.exists())
print('MRO exists       :', ECB_POLICY_RATES_CSV.exists())
print('EUROSTOXX exists :', EUROSTOXX_CSV.exists())

[PATHS]
DATA_DIR         : C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset
STATEMENTS exists: True
MRO exists       : True
EUROSTOXX exists : True


In [2]:
import csv

def _normalize_index_to_date_index(df, date_col='date'):
    """Normalize to a tz-naive date index with unique, sorted dates."""
    if date_col in df.columns:
        s = pd.to_datetime(df[date_col], errors='coerce')
    else:
        s = pd.to_datetime(df.index, errors='coerce')
    s = s.dt.tz_localize(None).dt.normalize()
    df = df.copy()
    df.index = s
    df = df[~df.index.isna()].sort_index()
    df = df[~df.index.duplicated(keep='first')]
    return df

def _sniff_csv_params(path, sample_bytes=65536):
    with open(path, 'rb') as f:
        raw = f.read(sample_bytes)
    encoding = 'utf-8-sig' if raw.startswith(b'\xef\xbb\xbf') else 'utf-8'
    try:
        text = raw.decode(encoding, errors='replace')
        dialect = csv.Sniffer().sniff(text, delimiters=[',',';','\t','|'])
        sep = dialect.delimiter
        quotechar = dialect.quotechar if dialect.quotechar else '"'
    except Exception:
        sep, quotechar = None, '"'
    return encoding, sep, quotechar

def load_ecb_statements(path=ECB_STATEMENTS_CSV):
    print(f"\n[LOAD] ECB statements from: {path}")
    enc, sep, quotechar = _sniff_csv_params(path)
    print(f"[SNIFF] encoding={enc} | sep={'auto' if sep is None else repr(sep)} | quotechar={repr(quotechar)}")
    tries = [
        dict(encoding=enc, sep=sep, engine='python', quotechar=quotechar, escapechar='\\'),
        dict(encoding=enc, sep=',', engine='python', quotechar='"', escapechar='\\'),
        dict(encoding=enc, sep=';', engine='python', quotechar='"', escapechar='\\'),
        dict(encoding='latin-1', sep=sep, engine='python', quotechar=quotechar, escapechar='\\'),
    ]
    df = None; last_err = None
    for i, kw in enumerate(tries, 1):
        try:
            df = pd.read_csv(path, **{k:v for k,v in kw.items() if v is not None})
            print(f"[READ OK] try#{i} -> shape={df.shape}")
            break
        except Exception as e:
            print(f"[READ FAIL] try#{i}: {e}")
            last_err = e
    if df is None:
        raise last_err
    if 'date' not in df.columns or 'text' not in df.columns:
        raise AssertionError("Statements CSV must contain 'date' and 'text'.")
    df = _normalize_index_to_date_index(df, 'date')
    df = df[['text']]
    print(f"[OK] statements: shape={df.shape} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def load_ecb_mro(path=ECB_POLICY_RATES_CSV):
    print(f"\n[LOAD] ECB MRO from: {path}")
    df = pd.read_csv(path)
    assert 'date' in df.columns and 'mro_rate' in df.columns, "MRO CSV must have 'date' and 'mro_rate'"
    df['mro_rate'] = pd.to_numeric(df['mro_rate'], errors='coerce')
    df = df.dropna(subset=['mro_rate'])
    df = _normalize_index_to_date_index(df, 'date')
    print(f"[OK] MRO daily: shape={df.shape} | unique rates={df['mro_rate'].nunique()} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def compute_rate_change_events(mro: pd.DataFrame):
    """Detect effective change dates (delta != 0). Return DataFrame 'events' with 'change' in {-1,+1}."""
    mro = mro.sort_index().copy()
    delta = mro['mro_rate'].diff()
    ev = delta[delta.fillna(0) != 0]
    events = pd.DataFrame(index=ev.index)
    events['change'] = np.sign(ev.values).astype(int)
    print(f"[EVENTS] changes: total={len(events)} | hikes={(events['change']==1).sum()} | cuts={(events['change']==-1).sum()}")
    return events

def subset_policy_statements(statements_csv_path, events_index,
                             keywords=(
                                 'monetary policy decision',
                                 'introductory statement',
                                 'press conference',
                                 'press release',
                                 'governing council',
                                 'interest rate')):
    """Optional narrowing to policy-relevant statements using 'title' keywords if a title column exists."""
    df = pd.read_csv(statements_csv_path)
    assert 'date' in df.columns and 'text' in df.columns, "CSV must have 'date' and 'text'"
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.tz_localize(None).dt.normalize()
    df = df.dropna(subset=['date'])
    if 'title' in df.columns:
        t = df['title'].astype(str).str.lower()
        mask = False
        for k in keywords:
            mask = mask | t.str.contains(k)
        df = df[mask].copy()
    weekend = df['date'].dt.dayofweek >= 5
    if weekend.any():
        df.loc[weekend, 'date'] = (df.loc[weekend, 'date'] - BDay(1)).dt.normalize()
    df = df.sort_values('date').drop_duplicates('date', keep='first').set_index('date')
    df = df[['text']]
    if len(events_index):
        start, end = events_index.min(), events_index.max()
        df = df.loc[(df.index >= start - BDay(10)) & (df.index <= end + BDay(10))]
    print(f"[FILTER] policy-like statements kept: {len(df)}")
    return df

def map_events_to_statements(statements: pd.DataFrame,
                             events: pd.DataFrame,
                             back_bdays=3, fwd_bdays=2, prefer_past=True):
    """Map each effective rate-change event to the nearest statement within a business-day window.
    Preference is for past statements at ties (meeting day usually precedes the effective day)."""
    s_idx = pd.DatetimeIndex(statements.index).sort_values()
    ev_idx = pd.DatetimeIndex(events.index).sort_values()
    chosen = {}
    for e in ev_idx:
        window = set([e])
        for k in range(1, back_bdays+1):
            window.add(e - BDay(k))
        for k in range(1, fwd_bdays+1):
            window.add(e + BDay(k))
        candidates = s_idx.intersection(pd.DatetimeIndex(sorted(window)))
        if len(candidates) == 0:
            continue
        def bd_dist(s, e):
            if s == e: return 0
            k = 0; cur = s
            if s < e:
                while cur < e: cur += BDay(1); k += 1
            else:
                while cur > e: cur -= BDay(1); k += 1
            return k
        bestS, bestKey = None, None
        for s in candidates:
            dist = bd_dist(s, e)
            tie  = 1 if (prefer_past and s <= e) else 0
            key  = (dist, -tie)
            if bestKey is None or key < bestKey:
                bestKey, bestS = key, s
        change = int(events.loc[e, 'change'])
        prev = chosen.get(bestS)
        if prev is None or bestKey < prev[:2]:
            chosen[bestS] = (bestKey[0], bestKey[1], change)
    if not chosen:
        print(f"[MAP] No event mapped with window [-{back_bdays}BD, +{fwd_bdays}BD].")
        return statements.iloc[0:0].copy()
    S_dates = sorted(chosen.keys())
    df_lbl = statements.loc[S_dates].copy()
    df_lbl['change'] = [chosen[s][2] for s in S_dates]
    vc = df_lbl['change'].value_counts().to_dict()
    print(f"[MAP] labeled={len(df_lbl)} / {len(statements)} ({len(df_lbl)/len(statements)*100:.1f}%) | class_balance={vc}")
    return df_lbl

def train_tfidf_logistic(df_labeled: pd.DataFrame):
    X, y = df_labeled['text'], df_labeled['change']
    if y.nunique() < 2 or len(y) < 5:
        print('[MODEL] Not enough labeled samples for classification.')
        return None, None
    est = Pipeline(steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, lowercase=True,
                                  strip_accents='unicode', stop_words='english',
                                  token_pattern=r'\b[a-zA-Z]{3,}\b')),
        ('log1p', FunctionTransformer(np.log1p, validate=False)),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                                   max_iter=5000, C=1.0, n_jobs=-1, random_state=0)),
    ])
    est.fit(X, y)
    vocab = np.array(sorted(est.named_steps['tfidf'].vocabulary_, key=lambda k: est.named_steps['tfidf'].vocabulary_[k]))
    coefs = est.named_steps['clf'].coef_[0]
    coef_df = pd.DataFrame(coefs, index=vocab, columns=['coef'])
    print('[MODEL] Logistic trained. n_features:', len(vocab))
    return est, coef_df

def train_tfidf_enet_reg(df_labeled: pd.DataFrame):
    X, y = df_labeled['text'], df_labeled['change']
    if y.nunique() < 2 or len(y) < 5:
        print('[MODEL] Not enough labeled samples for regression.')
        return None, None
    est = Pipeline(steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, lowercase=True,
                                  strip_accents='unicode', stop_words='english',
                                  token_pattern=r'\b[a-zA-Z]{3,}\b')),
        ('reg', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=0)),
    ])
    est.fit(X, y)
    vocab = np.array(sorted(est.named_steps['tfidf'].vocabulary_, key=lambda k: est.named_steps['tfidf'].vocabulary_[k]))
    coefs = est.named_steps['reg'].coef_
    coef_df = pd.DataFrame(coefs, index=vocab, columns=['coef'])
    print('[MODEL] ElasticNet trained. n_features:', len(vocab))
    return est, coef_df

In [3]:
# --- Run end-to-end
statements_all = load_ecb_statements()
mro_daily      = load_ecb_mro()
events         = compute_rate_change_events(mro_daily)

# Optional narrowing to policy-relevant items (uses 'title' keywords if present)
statements_pol = subset_policy_statements(ECB_STATEMENTS_CSV, events.index)

# Map events → statements (tune window if needed)
df_labeled = map_events_to_statements(statements_pol, events, back_bdays=3, fwd_bdays=2, prefer_past=True)

print('\n[SUMMARY]')
print('  statements (all) :', len(statements_all))
print('  policy-like      :', len(statements_pol))
print('  labeled (±1)     :', len(df_labeled))
print('  unlabeled        :', len(statements_pol) - len(df_labeled))

# Train models (only if enough labels)
log_est, log_coef = train_tfidf_logistic(df_labeled)
if log_coef is not None:
    coefs_plot(log_coef, title='TF-IDF + Logistic (elastic-net) — coefficients')

enet_est, enet_coef = train_tfidf_enet_reg(df_labeled)
if enet_coef is not None:
    coefs_plot(enet_coef, title='TF-IDF + ElasticNet (regression) — coefficients')


[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 1) | span=1997-02-07→2025-09-30

[LOAD] ECB MRO from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_policy_rates_daily_fake.csv
[OK] MRO daily: shape=(2827, 2) | unique rates=6 | span=2015-01-01→2025-10-31
[EVENTS] changes: total=11 | hikes=8 | cuts=3
[FILTER] policy-like statements kept: 26
[MAP] labeled=1 / 26 (3.8%) | class_balance={1: 1}

[SUMMARY]
  statements (all) : 2249
  policy-like      : 26
  labeled (±1)     : 1
  unlabeled        : 25
[MODEL] Not enough labeled samples for classification.
[MODEL] Not enough labeled samples for regression.
